In [2]:
pip install selenium --upgrade


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

df = pd.read_excel("all_data.xlsx")
print("📌 Columns found:", df.columns.tolist())


📌 Columns found: ['TDR', 'Tendering Authority', 'Tender Brief', 'DocumentFees', 'EMD', 'Tender Amount', 'Location', 'State', 'DueDate', 'TenderNo', 'Address', 'ContactEmail', 'TenderId', 'pubDate']


In [5]:
import pandas as pd

# Load the Excel file
df = pd.read_excel("all_data.xlsx")

# Use the correct column name
tdr_ids = df['TenderId'].dropna().astype(str).tolist()

print("✅ Number of TDR IDs found:", len(tdr_ids))
print("🔢 Sample TDR IDs:", tdr_ids[:5])  # show first 5


✅ Number of TDR IDs found: 59020
🔢 Sample TDR IDs: ['2025_PHEO_113334_1', '2025_KWA_759976_1', '2025_MAD_835433_1', '2025_PHEO_111048_1', '2025_MAD_812211_1']


In [8]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Load TDR IDs from the previous step
df = pd.read_excel("all_data.xlsx")
tdr_ids = df['TenderId'].dropna().astype(str).tolist()

# Prepare output
output = []

# Setup Selenium
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

for tdr in tdr_ids[:10]:  # Limit for testing (remove [:10] to run all)
    try:
        url = f"https://www.tenderdetail.com/registeruser/searchtenders?ServiceType=1&nm={tdr}"
        driver.get(url)
        time.sleep(3)  # Wait for page to load

        # Extract key fields (you can adjust these based on what appears)
        tender_title = driver.find_element(By.XPATH, "//h1").text.strip()
        rows = driver.find_elements(By.XPATH, "//div[@class='tender-detail-data']//tr")

        tender_data = {'TDR ID': tdr, 'Title': tender_title}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) == 2:
                key = cols[0].text.strip()
                val = cols[1].text.strip()
                tender_data[key] = val

        output.append(tender_data)
        print(f"✅ Extracted {tdr}")
    
    except Exception as e:
        print(f"❌ Failed {tdr}: {e}")
        continue

driver.quit()

# Save results
out_df = pd.DataFrame(output)
out_df.to_excel("tender_details_output.xlsx", index=False)
print("💾 Results saved to tender_details_output.xlsx")


✅ Extracted 2025_PHEO_113334_1
✅ Extracted 2025_KWA_759976_1
✅ Extracted 2025_MAD_835433_1
✅ Extracted 2025_PHEO_111048_1
✅ Extracted 2025_MAD_812211_1
✅ Extracted RVU2526WSOB00431
✅ Extracted 192913
✅ Extracted 2025_CDSNW_1044920_1
✅ Extracted 2025_KSCSB_1032587_3
✅ Extracted 602801
💾 Results saved to tender_details_output.xlsx
